# 00 – Transform Raw Firewall Logs → Parquet

This notebook demonstrates how to:
1. Parse raw Linux kernel netfilter / iptables syslog lines using `KernelFirewallParser`
2. Inspect the structured DataFrame
3. Persist the data to **Parquet** format using `save_parquet` from `utils.io`
4. Reload and verify the Parquet file

The parser and I/O helpers used here are fully reusable from any part of the application.

In [1]:
import sys
from pathlib import Path

# Make the project root importable when running from the notebooks/ folder
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE


## 1 – Configuration des chemins

Renseignez ici le chemin vers votre fichier `.log` brut et le chemin de sortie Parquet.

`LOG_YEAR` est l'**année de départ** des logs (la première entrée du fichier).  
Le parser détecte automatiquement les changements d'année en surveillant les retours en arrière dans la séquence des mois (ex. décembre → janvier), ce qui permet de traiter des fichiers qui couvrent plusieurs années calendaires.

In [2]:
from pathlib import Path

# ── À MODIFIER ────────────────────────────────────────────────────────────────
LOG_PATH     = "C:/Users/cyraptor/Downloads/log_brute.log/log_brute.log"   # chemin du .log brut
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "firewall_sample.parquet"  # sortie .parquet
LOG_YEAR     = 2025   # année de la PREMIÈRE entrée du fichier (les changements suivants sont auto-détectés)

# Filtre de dates (optionnel) — mettre FILTER_DATES = False pour désactiver
FILTER_DATES = True
DATE_FROM    = "2025-11-01"   # inclus
DATE_TO      = "2026-02-28"   # inclus
# ──────────────────────────────────────────────────────────────────────────────

# Vérifie que le fichier source existe
if not Path(LOG_PATH).exists():
    raise FileNotFoundError(f"Fichier introuvable : {LOG_PATH}")

print(f"Source          : {LOG_PATH}")
print(f"Sortie          : {PARQUET_PATH}")
print(f"Année de départ : {LOG_YEAR}")
if FILTER_DATES:
    print(f"Filtre dates    : {DATE_FROM}  →  {DATE_TO}")

Source          : C:/Users/cyraptor/Downloads/log_brute.log/log_brute.log
Sortie          : C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_sample.parquet
Année de départ : 2025
Filtre dates    : 2025-11-01  →  2026-02-28


## 2 – Parser les logs bruts avec `KernelFirewallParser`

Le parser utilise `LOG_PATH` et `LOG_YEAR` définis en section 1.  
Il est également utilisable directement dans l'application :

```python
from parsers import KernelFirewallParser
# ou via la factory :
from parsers import ParserFactory
parser = ParserFactory.create("kernel_firewall", year=2025)
df = parser.parse("/chemin/vers/firewall.log")
```

In [3]:
from parsers import KernelFirewallParser

parser = KernelFirewallParser(year=LOG_YEAR)

df = parser.parse(str(LOG_PATH))

is_valid, errors = parser.validate(df)
print(f"Valid: {is_valid}  |  Errors: {errors or 'none'}")
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df

Valid: True  |  Errors: none

Shape: (11997652, 18)
Columns: ['timestamp', 'hostname', 'action', 'fw', 'rule', 'interface_in', 'interface_out', 'src_ip', 'dst_ip', 'len', 'ttl', 'id', 'df', 'proto', 'src_port', 'dst_port', 'window', 'flags']


,timestamp,hostname,action,fw,rule,interface_in,interface_out,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2025-03-20 01:29:24,159.84.146.99,DENY,6,999,eth0,,94.102.61.47,159.84.146.99,44,238,54321,False,TCP,52502,3178,65535,SYN
1,2025-03-20 01:29:25,159.84.146.99,DENY,6,999,eth0,,176.111.174.85,159.84.146.99,40,229,16747,False,TCP,48739,2231,1024,SYN
2,2025-03-20 01:29:27,159.84.146.99,PERMIT,6,1,eth0,,66.249.65.106,159.84.146.99,60,114,49674,False,TCP,50501,443,65535,SYN
3,2025-03-20 01:29:34,159.84.146.99,DENY,6,999,eth0,,89.248.163.75,159.84.146.99,40,237,51442,False,TCP,43312,8845,1024,SYN
4,2025-03-20 01:29:38,159.84.146.99,DENY,6,7,eth0,,42.58.163.244,159.84.146.99,40,44,32986,False,TCP,9746,23,54266,SYN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11997647,2026-02-12 10:08:30,159.84.146.99,PERMIT,6,1,eth0,,23.22.35.162,159.84.146.99,60,102,52611,True,TCP,12680,443,62727,SYN
11997648,2026-02-12 10:08:33,159.84.146.99,PERMIT,6,1,eth0,,114.45.140.91,159.84.146.99,60,48,4438,True,TCP,42640,443,64240,SYN
11997649,2026-02-12 10:08:34,159.84.146.99,PERMIT,6,1,eth0,,3.224.220.101,159.84.146.99,60,110,29437,True,TCP,24581,443,62727,SYN
11997650,2026-02-12 10:08:36,159.84.146.99,PERMIT,6,1,eth0,,192.44.68.176,159.84.146.99,60,44,9693,True,TCP,13575,443,40520,SYN


In [4]:
# Inspect dtypes and a quick summary
print(df.dtypes)
print()
df[["timestamp", "action", "src_ip", "dst_ip", "src_port", "dst_port", "rule", "flags"]].head(10)

timestamp        datetime64[ns]
hostname                 object
action                 category
fw                        Int64
rule                      Int64
interface_in             object
interface_out            object
src_ip                   object
dst_ip                   object
len                       Int64
ttl                       Int64
id                        Int64
df                         bool
proto                  category
src_port                  Int64
dst_port                  Int64
window                    Int64
flags                    object
dtype: object



,timestamp,action,src_ip,dst_ip,src_port,dst_port,rule,flags
0,2025-03-20 01:29:24,DENY,94.102.61.47,159.84.146.99,52502,3178,999,SYN
1,2025-03-20 01:29:25,DENY,176.111.174.85,159.84.146.99,48739,2231,999,SYN
2,2025-03-20 01:29:27,PERMIT,66.249.65.106,159.84.146.99,50501,443,1,SYN
3,2025-03-20 01:29:34,DENY,89.248.163.75,159.84.146.99,43312,8845,999,SYN
4,2025-03-20 01:29:38,DENY,42.58.163.244,159.84.146.99,9746,23,7,SYN
5,2025-03-20 01:29:39,PERMIT,173.252.83.5,159.84.146.99,50908,443,1,SYN
6,2025-03-20 01:29:41,PERMIT,66.249.65.106,159.84.146.99,45634,443,1,SYN
7,2025-03-20 01:29:48,PERMIT,66.249.65.106,159.84.146.99,40785,443,1,SYN
8,2025-03-20 01:29:53,PERMIT,34.30.180.51,159.84.146.99,59756,443,1,SYN
9,2025-03-20 01:29:58,DENY,85.217.144.149,159.84.146.99,43267,33944,999,SYN


## 3 – Post-traitement : filtre de dates & suppression de `fw`

Conformément aux consignes :
- la colonne `fw` (numéro du firewall) est **supprimée** — dans le sujet, il y est indiqué que cette information n'est pas pertinente pour l'analyse
- seules les lignes entre **novembre 2025** et **février 2026** sont conservées (si `FILTER_DATES = True`)

In [5]:
df_out = df.copy()

# ── Distribution des valeurs FW (informatif) ───────────────────────────────
print("Distribution des valeurs FW (toutes conservées) :")
print(df_out["fw"].value_counts().sort_index().to_string())
print()

# ── Suppression de la colonne fw (consigne : information à supprimer) ──────
df_out = df_out.drop(columns=["fw"], errors="ignore")

# ── Filtre de dates (optionnel) ────────────────────────────────────────────
if FILTER_DATES:
    before = len(df_out)
    mask   = (df_out["timestamp"] >= DATE_FROM) & (df_out["timestamp"] <= DATE_TO)
    df_out = df_out.loc[mask].reset_index(drop=True)
    print(f"Filtre dates : {before:,} → {len(df_out):,} lignes  ({DATE_FROM} → {DATE_TO})")
else:
    print(f"Filtre de dates désactivé — {len(df_out):,} lignes conservées")

print(f"\nColonnes : {list(df_out.columns)}")
df_out.head(5)

Distribution des valeurs FW (toutes conservées) :
fw
1    2405006
6    9592646

Filtre dates : 11,997,652 → 4,572,903 lignes  (2025-11-01 → 2026-02-28)

Colonnes : ['timestamp', 'hostname', 'action', 'rule', 'interface_in', 'interface_out', 'src_ip', 'dst_ip', 'len', 'ttl', 'id', 'df', 'proto', 'src_port', 'dst_port', 'window', 'flags']


,timestamp,hostname,action,rule,interface_in,interface_out,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2025-11-12 09:56:37,159.84.146.99,DENY,999,eth0,,77.90.185.64,159.84.146.99,40,236,63864,False,TCP,55366,65153,1024,SYN
1,2025-11-12 09:56:37,159.84.146.99,PERMIT,1,eth0,,47.128.20.252,159.84.146.99,60,33,13061,True,TCP,14436,443,35844,SYN
2,2025-11-12 09:56:38,159.84.146.99,PERMIT,1,eth0,,23.22.35.162,159.84.146.99,60,102,28591,True,TCP,55973,443,62727,SYN
3,2025-11-12 09:56:39,159.84.146.99,DENY,999,eth0,,79.124.60.150,159.84.146.99,40,236,20551,False,TCP,53744,56283,1024,SYN
4,2025-11-12 09:56:39,159.84.146.99,PERMIT,1,eth0,,47.128.20.252,159.84.146.99,60,32,15224,True,TCP,25404,443,35844,SYN


## 4 – Sauvegarde en Parquet

`save_parquet` et `load_parquet` sont dans `utils.io` et peuvent être importés partout dans l'application.

In [6]:
from utils.io import save_parquet, load_parquet

saved = save_parquet(df_out, PARQUET_PATH, compression="snappy")
print(f"Sauvegardé → {saved}  ({saved.stat().st_size:,} bytes)")

Sauvegardé → C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_sample.parquet  (67,331,926 bytes)


## 5 – Reload & vérification

In [7]:
df_loaded = load_parquet(PARQUET_PATH)

print(f"Rows loaded: {len(df_loaded)}")
print(f"Dtypes preserved: {dict(df_loaded.dtypes)}")
df_loaded

Rows loaded: 4572903
Dtypes preserved: {'timestamp': dtype('<M8[ns]'), 'hostname': dtype('O'), 'action': CategoricalDtype(categories=['DENY', 'PERMIT'], ordered=False, categories_dtype=object), 'rule': Int64Dtype(), 'interface_in': dtype('O'), 'interface_out': dtype('O'), 'src_ip': dtype('O'), 'dst_ip': dtype('O'), 'len': Int64Dtype(), 'ttl': Int64Dtype(), 'id': Int64Dtype(), 'df': dtype('bool'), 'proto': CategoricalDtype(categories=['', 'TCP'], ordered=False, categories_dtype=object), 'src_port': Int64Dtype(), 'dst_port': Int64Dtype(), 'window': Int64Dtype(), 'flags': dtype('O')}


,timestamp,hostname,action,rule,interface_in,interface_out,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2025-11-12 09:56:37,159.84.146.99,DENY,999,eth0,,77.90.185.64,159.84.146.99,40,236,63864,False,TCP,55366,65153,1024,SYN
1,2025-11-12 09:56:37,159.84.146.99,PERMIT,1,eth0,,47.128.20.252,159.84.146.99,60,33,13061,True,TCP,14436,443,35844,SYN
2,2025-11-12 09:56:38,159.84.146.99,PERMIT,1,eth0,,23.22.35.162,159.84.146.99,60,102,28591,True,TCP,55973,443,62727,SYN
3,2025-11-12 09:56:39,159.84.146.99,DENY,999,eth0,,79.124.60.150,159.84.146.99,40,236,20551,False,TCP,53744,56283,1024,SYN
4,2025-11-12 09:56:39,159.84.146.99,PERMIT,1,eth0,,47.128.20.252,159.84.146.99,60,32,15224,True,TCP,25404,443,35844,SYN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4572898,2026-02-12 10:08:30,159.84.146.99,PERMIT,1,eth0,,23.22.35.162,159.84.146.99,60,102,52611,True,TCP,12680,443,62727,SYN
4572899,2026-02-12 10:08:33,159.84.146.99,PERMIT,1,eth0,,114.45.140.91,159.84.146.99,60,48,4438,True,TCP,42640,443,64240,SYN
4572900,2026-02-12 10:08:34,159.84.146.99,PERMIT,1,eth0,,3.224.220.101,159.84.146.99,60,110,29437,True,TCP,24581,443,62727,SYN
4572901,2026-02-12 10:08:36,159.84.146.99,PERMIT,1,eth0,,192.44.68.176,159.84.146.99,60,44,9693,True,TCP,13575,443,40520,SYN
